# AED v4 — corrigido, balanceado e estável no Colab

Este notebook substitui o experimento anterior e corrige os problemas encontrados:

- áudios da pasta `scream` não entram mais como `ambiente_normal`;
- o caminho raiz contendo `scream-and-non-scream` não interfere no rótulo;
- tiros são agrupados por tipo de arma, reduzindo vazamento;
- UrbanSound8K é agrupado pelo áudio original;
- split ocorre por grupo antes de qualquer augmentation;
- features são gravadas em disco, não acumuladas em arrays gigantes;
- treinamento usa `tf.data` por streaming;
- dataset balanceado por classe com amostragem uniforme;
- dataset infinito é usado somente no treino, com `steps_per_epoch`;
- validação e teste permanecem finitos;
- macro-F1 é calculado durante a própria validação;
- cabeça e ajuste fino são comparados;
- exportação TFLite permanece com uma entrada única;
- paridade Keras × TFLite é verificada;
- falsos alertas em áudios realmente normais são medidos.

Classes:

1. `ambiente_normal`
2. `grito_panico`
3. `choro_distress`
4. `voz_agressiva`
5. `impacto`
6. `vidro_quebrando`
7. `tiro`

A meta de 90% é experimental. O notebook não inventa resultado nem aceita vazamento para alcançar a meta.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install             kagglehub             librosa==0.11.0             soundfile             scikit-learn             pandas             matplotlib             tqdm             pyyaml             webrtcvad-wheels             requests

In [ ]:
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import gc
import hashlib
import json
import math
import random
import re
import shutil
import subprocess
import zipfile

import cv2
import kagglehub
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import soundfile as sf
import tensorflow as tf
import webrtcvad

from google.colab import files
from IPython.display import Audio, display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from tqdm.auto import tqdm

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError(
        "Ative uma GPU T4 no Colab antes de continuar."
    )

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True,
        )
    except RuntimeError:
        pass

tf.keras.mixed_precision.set_global_policy(
    "mixed_float16"
)

print(
    "Política de precisão:",
    tf.keras.mixed_precision.global_policy(),
)

## 1. Configuração

In [ ]:
@dataclass(frozen=True)
class ConfiguracaoAED:
    sr: int = 16000
    duracao: float = 2.0
    passo_treino: float = 1.0
    passo_inferencia: float = 0.5

    n_fft: int = 512
    hop_length: int = 256
    n_mels: int = 96
    n_mfcc: int = 40
    imagem: int = 128

    max_arquivos_por_classe: int = 3000

    epocas_cabeca: int = 18
    epocas_ajuste: int = 25
    lote_cabeca: int = 16
    lote_ajuste: int = 8
    lote_validacao: int = 32
    paciencia: int = 7

    passos_minimos_epoca: int = 500
    passos_maximos_epoca: int = 1000

    seed: int = 42
    minimo_consecutivas_alerta: int = 2

    meta_acuracia: float = 0.90
    meta_f1_macro: float = 0.85
    meta_paridade_tflite: float = 0.99
    max_falsos_alertas_hora: float = 30.0

    usar_urban8k: bool = True
    usar_scream_dataset: bool = True
    usar_tut: bool = True
    usar_reforco_manual: bool = False


CFG = ConfiguracaoAED()

CLASSES = [
    "ambiente_normal",
    "grito_panico",
    "choro_distress",
    "voz_agressiva",
    "impacto",
    "vidro_quebrando",
    "tiro",
]

ID = {
    classe: indice
    for indice, classe in enumerate(CLASSES)
}

JANELAS_POR_FONTE = {
    "ambiente_normal": 2,
    "grito_panico": 3,
    "choro_distress": 3,
    "voz_agressiva": 3,
    "impacto": 5,
    "vidro_quebrando": 5,
    "tiro": 3,
}

RAIZ = Path("/content/aed_v4")
DADOS = RAIZ / "datasets"
FEATURES = RAIZ / "features"
MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados"
ENTREGA = RAIZ / "entrega"

ESC_DIR = DADOS / "esc50"
TUT_DIR = DADOS / "tut_rare"
REFORCO_MANUAL = DADOS / "reforco_manual"

for pasta in (
    RAIZ,
    DADOS,
    FEATURES,
    MODELOS,
    RESULTADOS,
    ENTREGA,
):
    pasta.mkdir(
        parents=True,
        exist_ok=True,
    )

random.seed(CFG.seed)
np.random.seed(CFG.seed)
tf.random.set_seed(CFG.seed)

display(pd.DataFrame([asdict(CFG)]))

## 2. Baixar datasets

In [ ]:
ASVP = Path(
    kagglehub.dataset_download(
        (
            "dejolilandry/"
            "asvpesdspeech-nonspeech-emotional-utterances"
        )
    )
)

GUNSHOT = Path(
    kagglehub.dataset_download(
        "emrahaydemr/gunshot-audio-dataset"
    )
)

URBAN = None

if CFG.usar_urban8k:
    URBAN = Path(
        kagglehub.dataset_download(
            "chrisfilo/urbansound8k"
        )
    )

SCREAM = None

if CFG.usar_scream_dataset:
    SCREAM = Path(
        kagglehub.dataset_download(
            (
                "aananehsansiam/"
                "audio-dataset-of-scream-and-non-scream"
            )
        )
    )

print("ASVP:", ASVP)
print("Gunshot:", GUNSHOT)
print("UrbanSound8K:", URBAN)
print("Scream/non-scream:", SCREAM)

In [ ]:
def zip_valido(
    caminho: Path,
) -> bool:
    return (
        caminho.exists()
        and caminho.stat().st_size > 0
        and zipfile.is_zipfile(caminho)
    )


def baixar_zip(
    urls: list[str],
    destino: Path,
) -> None:
    if zip_valido(destino):
        print("Já disponível:", destino)
        return

    destino.unlink(
        missing_ok=True
    )

    temporario = destino.with_suffix(
        destino.suffix + ".part"
    )
    temporario.unlink(
        missing_ok=True
    )

    ultimo_erro = None

    for url in urls:
        for verificar_ssl in (
            True,
            False,
        ):
            try:
                print(
                    "Baixando:",
                    url,
                    "| SSL:",
                    verificar_ssl,
                )

                with requests.get(
                    url,
                    stream=True,
                    timeout=(30, 180),
                    verify=verificar_ssl,
                    allow_redirects=True,
                    headers={
                        "User-Agent": (
                            "Mozilla/5.0 "
                            "violence-edge-ai-aed"
                        )
                    },
                ) as resposta:
                    resposta.raise_for_status()

                    total = int(
                        resposta.headers.get(
                            "content-length",
                            0,
                        )
                    )

                    with temporario.open(
                        "wb"
                    ) as arquivo:
                        with tqdm(
                            total=total,
                            unit="B",
                            unit_scale=True,
                            desc=destino.name,
                        ) as barra:
                            for bloco in resposta.iter_content(
                                chunk_size=1024 * 1024
                            ):
                                if not bloco:
                                    continue

                                arquivo.write(
                                    bloco
                                )
                                barra.update(
                                    len(bloco)
                                )

                temporario.replace(
                    destino
                )

                if not zip_valido(
                    destino
                ):
                    destino.unlink(
                        missing_ok=True
                    )
                    raise RuntimeError(
                        "O arquivo baixado não é "
                        "um ZIP válido."
                    )

                return

            except Exception as erro:
                ultimo_erro = erro
                temporario.unlink(
                    missing_ok=True
                )
                print(
                    "Tentativa falhou:",
                    type(erro).__name__,
                    erro,
                )

    raise RuntimeError(
        "Falha ao baixar o ZIP. "
        f"Último erro: {ultimo_erro}"
    )


def extrair_zip(
    arquivo_zip: Path,
    destino: Path,
) -> None:
    marcador = (
        destino / ".extraido"
    )

    if marcador.exists():
        return

    destino.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        arquivo_zip
    ) as arquivo:
        arquivo.extractall(
            destino
        )

    marcador.touch()


ZIP_ESC = DADOS / "esc50.zip"

baixar_zip(
    [
        (
            "https://github.com/karolpiczak/"
            "ESC-50/archive/refs/heads/master.zip"
        )
    ],
    ZIP_ESC,
)
extrair_zip(
    ZIP_ESC,
    ESC_DIR,
)

if CFG.usar_tut:
    ZIP_TUT = (
        DADOS / "tut_eventos.zip"
    )

    baixar_zip(
        [
            (
                "https://zenodo.org/records/401395/files/"
                "TUT-rare-sound-events-2017-development."
                "source_data_events.zip?download=1"
            )
        ],
        ZIP_TUT,
    )
    extrair_zip(
        ZIP_TUT,
        TUT_DIR,
    )

print("ESC-50:", ESC_DIR)
print("TUT:", TUT_DIR)

## 3. Reforço manual opcional

O ZIP opcional deve conter pastas com os nomes das classes:

```text
reforco/
├── ambiente_normal/
├── grito_panico/
├── choro_distress/
├── voz_agressiva/
├── impacto/
├── vidro_quebrando/
└── tiro/
```

Deixe `usar_reforco_manual=False` para seguir sem upload.

In [ ]:
if CFG.usar_reforco_manual:
    enviados = files.upload()

    arquivos_zip = [
        nome
        for nome in enviados
        if nome.lower().endswith(".zip")
    ]

    if len(arquivos_zip) != 1:
        raise RuntimeError(
            "Envie exatamente um ZIP."
        )

    if REFORCO_MANUAL.exists():
        shutil.rmtree(
            REFORCO_MANUAL
        )

    REFORCO_MANUAL.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        arquivos_zip[0]
    ) as arquivo:
        arquivo.extractall(
            REFORCO_MANUAL
        )

    print(
        "Reforço extraído:",
        REFORCO_MANUAL,
    )
else:
    print("Reforço manual desativado.")

## 4. Construir metadados corrigidos

In [ ]:
EXTENSOES_AUDIO = {
    ".wav",
    ".mp3",
    ".ogg",
    ".flac",
    ".m4a",
}


def audios_em(
    pasta: Path | None,
) -> list[Path]:
    if pasta is None or not pasta.exists():
        return []

    return sorted(
        caminho
        for caminho in pasta.rglob("*")
        if (
            caminho.is_file()
            and caminho.suffix.lower()
            in EXTENSOES_AUDIO
        )
    )


def normalizar_token(
    texto: str,
) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(texto).lower(),
    ).strip("_")


def adicionar(
    registros: list[dict[str, Any]],
    arquivo: Path,
    classe: str,
    fonte: str,
    grupo: str,
    origem_id: str = "",
    extra: str = "",
) -> None:
    arquivo = Path(arquivo)

    if (
        arquivo.exists()
        and classe in ID
    ):
        registros.append(
            {
                "arquivo": str(
                    arquivo.resolve()
                ),
                "classe": classe,
                "fonte": fonte,
                "grupo": str(grupo),
                "origem_id": str(
                    origem_id
                ),
                "extra": str(extra),
            }
        )


registros = []


# ------------------------------------------------------------
# ASVP-ESD
# emoção = terceiro identificador
# ator = sexto identificador
# ------------------------------------------------------------

MAPA_ASVP = {
    "02": "ambiente_normal",
    "04": "choro_distress",
    "05": "voz_agressiva",
    "06": "grito_panico",
}

contagem_asvp = Counter()

for arquivo in audios_em(ASVP):
    numeros = re.findall(
        r"\d+",
        arquivo.stem,
    )

    if len(numeros) < 6:
        continue

    codigo_emocao = (
        numeros[2].zfill(2)
    )
    codigo_ator = (
        numeros[5].zfill(3)
    )

    classe = MAPA_ASVP.get(
        codigo_emocao
    )

    if classe is None:
        continue

    adicionar(
        registros,
        arquivo,
        classe,
        "ASVP",
        f"asvp:ator_{codigo_ator}",
        arquivo.stem,
        codigo_emocao,
    )

    contagem_asvp[
        classe
    ] += 1

print(
    "ASVP:",
    dict(contagem_asvp),
)

if (
    contagem_asvp["voz_agressiva"]
    == 0
):
    raise RuntimeError(
        "ASVP sem voz_agressiva."
    )


# ------------------------------------------------------------
# Gunshot
# grupo por pasta da arma, não por arquivo individual
# ------------------------------------------------------------

for arquivo in audios_em(GUNSHOT):
    try:
        relativo = arquivo.relative_to(
            GUNSHOT
        )
        arma = normalizar_token(
            relativo.parts[0]
        )
    except Exception:
        arma = normalizar_token(
            arquivo.parent.name
        )

    adicionar(
        registros,
        arquivo,
        "tiro",
        "GunshotKaggle",
        f"gunshot:arma_{arma}",
        arquivo.stem,
        arma,
    )


# ------------------------------------------------------------
# ESC-50
# ------------------------------------------------------------

csvs_esc = list(
    ESC_DIR.rglob(
        "esc50.csv"
    )
)
pastas_audio_esc = [
    pasta
    for pasta in ESC_DIR.rglob(
        "audio"
    )
    if pasta.is_dir()
]

if (
    not csvs_esc
    or not pastas_audio_esc
):
    raise RuntimeError(
        "ESC-50 incompleto."
    )

esc = pd.read_csv(
    csvs_esc[0]
)
esc_audio = (
    pastas_audio_esc[0]
)

MAPA_ESC = {
    "crying_baby": (
        "choro_distress"
    ),
    "glass_breaking": (
        "vidro_quebrando"
    ),
    "door_wood_knock": (
        "impacto"
    ),
    "door_wood_creaks": (
        "impacto"
    ),
    "clapping": "impacto",
    "hand_saw": "impacto",
}

NORMAIS_ESC = {
    "rain",
    "sea_waves",
    "crickets",
    "chirping_birds",
    "water_drops",
    "wind",
    "footsteps",
    "laughing",
    "breathing",
    "snoring",
    "clock_tick",
    "washing_machine",
    "vacuum_cleaner",
    "engine",
    "train",
    "airplane",
    "helicopter",
    "thunderstorm",
    "siren",
    "keyboard_typing",
    "mouse_click",
    "can_opening",
    "brushing_teeth",
    "fireworks",
    "chainsaw",
}

for linha in esc.itertuples():
    categoria = str(
        linha.category
    )

    if categoria in MAPA_ESC:
        classe = MAPA_ESC[
            categoria
        ]
    elif categoria in NORMAIS_ESC:
        classe = (
            "ambiente_normal"
        )
    else:
        continue

    adicionar(
        registros,
        (
            esc_audio
            / str(linha.filename)
        ),
        classe,
        "ESC-50",
        f"esc50:src_{linha.src_file}",
        linha.filename,
        categoria,
    )


# ------------------------------------------------------------
# TUT Rare
# ------------------------------------------------------------

if CFG.usar_tut:
    for arquivo in audios_em(
        TUT_DIR
    ):
        pasta_evento = (
            normalizar_token(
                arquivo.parent.name
            )
        )

        if pasta_evento in {
            "babycry",
            "baby_cry",
        }:
            classe = (
                "choro_distress"
            )

        elif pasta_evento in {
            "glassbreak",
            "glass_break",
        }:
            classe = (
                "vidro_quebrando"
            )

        elif pasta_evento in {
            "gunshot",
            "gun_shot",
        }:
            classe = "tiro"

        else:
            continue

        adicionar(
            registros,
            arquivo,
            classe,
            "TUT_Rare",
            (
                f"tut:{pasta_evento}:"
                f"{arquivo.stem}"
            ),
            arquivo.stem,
            pasta_evento,
        )


# ------------------------------------------------------------
# UrbanSound8K
# grupo por fsID do áudio original
# ------------------------------------------------------------

if URBAN is not None:
    csvs_urban = list(
        URBAN.rglob(
            "UrbanSound8K.csv"
        )
    )

    if csvs_urban:
        tabela_urban = pd.read_csv(
            csvs_urban[0]
        )
        raiz_audio = (
            csvs_urban[0]
            .parent
            .parent
            / "audio"
        )

        MAPA_URBAN = {
            "gun_shot": "tiro",
            "jackhammer": "impacto",
            "drilling": "impacto",
            "air_conditioner": (
                "ambiente_normal"
            ),
            "children_playing": (
                "ambiente_normal"
            ),
            "street_music": (
                "ambiente_normal"
            ),
            "engine_idling": (
                "ambiente_normal"
            ),
            "car_horn": (
                "ambiente_normal"
            ),
            "siren": (
                "ambiente_normal"
            ),
            "dog_bark": (
                "ambiente_normal"
            ),
        }

        for _, linha in (
            tabela_urban.iterrows()
        ):
            nome_classe = str(
                linha["class"]
            )
            classe = MAPA_URBAN.get(
                nome_classe
            )

            if classe is None:
                continue

            arquivo = (
                raiz_audio
                / f"fold{int(linha['fold'])}"
                / str(
                    linha[
                        "slice_file_name"
                    ]
                )
            )

            fsid = str(
                linha.get(
                    "fsID",
                    str(
                        linha[
                            "slice_file_name"
                        ]
                    ).split("-")[0],
                )
            )

            adicionar(
                registros,
                arquivo,
                classe,
                "UrbanSound8K",
                f"urban:fsid_{fsid}",
                str(
                    linha[
                        "slice_file_name"
                    ]
                ),
                nome_classe,
            )


# ------------------------------------------------------------
# Scream / non-scream
# CORREÇÃO: usa apenas as pastas relativas ao dataset.
# O nome do dataset raiz não participa da decisão.
# ------------------------------------------------------------

contagem_scream = Counter()

if SCREAM is not None:
    for arquivo in audios_em(
        SCREAM
    ):
        try:
            relativo = arquivo.relative_to(
                SCREAM
            )
            pastas_relativas = [
                normalizar_token(
                    parte
                )
                for parte in (
                    relativo.parts[:-1]
                )
            ]
        except Exception:
            pastas_relativas = [
                normalizar_token(
                    arquivo.parent.name
                )
            ]

        classe = None

        if any(
            pasta in {
                "non_scream",
                "nonscream",
                "no_scream",
                "not_scream",
            }
            for pasta in pastas_relativas
        ):
            classe = (
                "ambiente_normal"
            )

        elif any(
            pasta in {
                "scream",
                "screams",
            }
            for pasta in pastas_relativas
        ):
            classe = (
                "grito_panico"
            )

        if classe is None:
            continue

        grupo_base = (
            re.sub(
                r"[_\-\s]*\d+$",
                "",
                arquivo.stem.lower(),
            )
            or arquivo.stem.lower()
        )

        adicionar(
            registros,
            arquivo,
            classe,
            "ScreamDataset",
            (
                f"scream:{classe}:"
                f"{normalizar_token(grupo_base)}:"
                f"{arquivo.stem}"
            ),
            arquivo.stem,
            "/".join(
                pastas_relativas
            ),
        )

        contagem_scream[
            classe
        ] += 1

print(
    "ScreamDataset:",
    dict(contagem_scream),
)

if SCREAM is not None:
    if (
        contagem_scream[
            "ambiente_normal"
        ] == 0
        or contagem_scream[
            "grito_panico"
        ] == 0
    ):
        raise RuntimeError(
            "O ScreamDataset precisa gerar "
            "amostras normais e de grito."
        )


# ------------------------------------------------------------
# Reforço manual opcional
# ------------------------------------------------------------

if CFG.usar_reforco_manual:
    for classe in CLASSES:
        pastas_classe = [
            pasta
            for pasta in (
                REFORCO_MANUAL.rglob(
                    classe
                )
            )
            if pasta.is_dir()
        ]

        for pasta in pastas_classe:
            for arquivo in audios_em(
                pasta
            ):
                adicionar(
                    registros,
                    arquivo,
                    classe,
                    "ReforcoManual",
                    (
                        f"manual:{classe}:"
                        f"{arquivo.stem}"
                    ),
                    arquivo.stem,
                )


meta = pd.DataFrame(
    registros
)

if meta.empty:
    raise RuntimeError(
        "Nenhum áudio foi encontrado."
    )

meta = meta.drop_duplicates(
    subset=[
        "arquivo",
        "classe",
    ]
)

partes = []

for classe in CLASSES:
    parte = meta[
        meta["classe"] == classe
    ].copy()

    if (
        len(parte)
        > CFG.max_arquivos_por_classe
    ):
        parte = parte.sample(
            CFG.max_arquivos_por_classe,
            random_state=(
                CFG.seed
                + ID[classe]
            ),
        )

    partes.append(
        parte
    )

meta = pd.concat(
    partes,
    ignore_index=True,
)

resumo_fontes = (
    meta.groupby(
        [
            "classe",
            "fonte",
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

resumo_geral = (
    meta.groupby("classe")
    .agg(
        arquivos=(
            "arquivo",
            "count",
        ),
        grupos=(
            "grupo",
            "nunique",
        ),
    )
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

display(resumo_fontes)
display(resumo_geral)

faltantes = [
    classe
    for classe in CLASSES
    if resumo_geral.loc[
        classe,
        "arquivos",
    ] == 0
]

if faltantes:
    raise RuntimeError(
        f"Classes sem dados: {faltantes}"
    )

poucos_grupos = {
    classe: int(
        resumo_geral.loc[
            classe,
            "grupos",
        ]
    )
    for classe in CLASSES
    if resumo_geral.loc[
        classe,
        "grupos",
    ] < 3
}

if poucos_grupos:
    raise RuntimeError(
        "Classes com menos de 3 grupos: "
        f"{poucos_grupos}"
    )

meta.to_csv(
    RESULTADOS
    / "fontes_unificadas.csv",
    index=False,
)

## 5. Auditoria manual rápida

In [ ]:
for classe in CLASSES:
    print("\n" + "=" * 72)
    print("Classe:", classe)

    parte = meta[
        meta["classe"] == classe
    ]

    amostras = parte.sample(
        min(
            2,
            len(parte),
        ),
        random_state=(
            CFG.seed + ID[classe]
        ),
    )

    for linha in (
        amostras.itertuples()
    ):
        print(
            linha.fonte,
            "|",
            Path(
                linha.arquivo
            ).name,
            "|",
            linha.extra,
        )
        display(
            Audio(
                linha.arquivo
            )
        )

In [ ]:
APROVAR_AUDITORIA = True

if not APROVAR_AUDITORIA:
    raise RuntimeError(
        "Ouça as amostras. Depois altere "
        "APROVAR_AUDITORIA para True."
    )

## 6. Split por grupos

In [ ]:
def dividir_grupos(
    grupos: list[str],
    semente: int,
):
    grupos = list(
        dict.fromkeys(grupos)
    )

    random.Random(
        semente
    ).shuffle(
        grupos
    )

    total = len(grupos)

    if total < 3:
        raise RuntimeError(
            "São necessários pelo menos "
            "3 grupos por classe."
        )

    n_treino = max(
        1,
        int(round(total * 0.70)),
    )
    n_validacao = max(
        1,
        int(round(total * 0.15)),
    )

    if (
        n_treino
        + n_validacao
        >= total
    ):
        n_treino = total - 2
        n_validacao = 1

    corte = (
        n_treino
        + n_validacao
    )

    return (
        set(
            grupos[:n_treino]
        ),
        set(
            grupos[
                n_treino:corte
            ]
        ),
        set(
            grupos[corte:]
        ),
    )


# Corrigindo o vazamento entre splits: dividir grupos globalmente uma única vez.

# 1. Obter todos os grupos únicos do DataFrame `meta`.
all_unique_grupos = meta["grupo"].drop_duplicates().tolist()

# 2. Dividir esses grupos globais em treino, validação e teste.
(
    global_grupos_treino,
    global_grupos_validacao,
    global_grupos_teste,
) = dividir_grupos(
    all_unique_grupos,
    CFG.seed, # Usar uma única semente para a divisão global
)

# 3. Criar um mapeamento de grupo para split
grupo_para_split = {}
for g in global_grupos_treino:
    grupo_para_split[g] = "treino"
for g in global_grupos_validacao:
    grupo_para_split[g] = "validacao"
for g in global_grupos_teste:
    grupo_para_split[g] = "teste"

# 4. Atribuir o split a cada linha do DataFrame `meta`
meta_split = meta.copy()
meta_split["split"] = meta_split["grupo"].map(grupo_para_split)

# Verificar se todos os grupos foram atribuídos (deve ser verdadeiro se all_unique_grupos for exaustivo)
if meta_split["split"].isnull().any():
    raise RuntimeError("Alguns grupos não foram atribuídos a um split.")

tabela_split = (
    meta_split.groupby(
        [
            "classe",
            "split",
        ]
    )["grupo"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

display(tabela_split)

for classe in CLASSES:
    for split in (
        "treino",
        "validacao",
        "teste",
    ):
        if (
            tabela_split.loc[
                classe,
                split,
            ]
            == 0
        ):
            raise RuntimeError(
                f"{classe} sem dados em {split}."
            )

vazamento = (
    meta_split.groupby(
        "grupo"
    )["split"]
    .nunique()
    .gt(1)
    .sum()
)

print(
    "Grupos presentes em mais "
    "de um split:",
    int(vazamento),
)

if vazamento:
    raise RuntimeError(
        "Vazamento entre splits."
    )

meta_split.to_csv(
    RESULTADOS
    / "fontes_com_split.csv",
    index=False,
)

## 7. Extrair features para arquivos PNG

Nenhum array gigante de treino é mantido na RAM.

A imagem contém:

- canal 1: log-mel;
- canal 2: MFCC;
- canal 3: delta do log-mel.

In [ ]:
AMOSTRAS = int(
    CFG.sr * CFG.duracao
)


def carregar_audio(
    caminho: str | Path,
) -> np.ndarray:
    audio, _ = librosa.load(
        str(caminho),
        sr=CFG.sr,
        mono=True,
    )

    audio = np.nan_to_num(
        audio.astype(
            np.float32
        )
    )

    pico = float(
        np.max(
            np.abs(audio)
        )
    )

    if pico > 1.0:
        audio = (
            audio / pico
        )

    return audio


def ajustar(
    audio: np.ndarray,
) -> np.ndarray:
    if len(audio) < AMOSTRAS:
        return np.pad(
            audio,
            (
                0,
                AMOSTRAS
                - len(audio),
            ),
        )

    return audio[
        :AMOSTRAS
    ]


def selecionar_janelas(
    audio: np.ndarray,
    classe: str,
) -> list[np.ndarray]:
    limite = (
        JANELAS_POR_FONTE[
            classe
        ]
    )

    if len(audio) <= AMOSTRAS:
        return [
            ajustar(audio)
        ]

    passo = int(
        CFG.sr
        * CFG.passo_treino
    )

    janelas = [
        audio[
            inicio:
            inicio + AMOSTRAS
        ]
        for inicio in range(
            0,
            len(audio)
            - AMOSTRAS
            + 1,
            passo,
        )
    ]

    if len(janelas) <= limite:
        return [
            ajustar(janela)
            for janela in janelas
        ]

    if (
        classe
        == "ambiente_normal"
    ):
        indices = np.linspace(
            0,
            len(janelas) - 1,
            limite,
            dtype=int,
        )
    else:
        energias = [
            float(
                np.sqrt(
                    np.mean(
                        janela ** 2
                    )
                    + 1e-10
                )
            )
            for janela in janelas
        ]

        indices = np.argsort(
            energias
        )[-limite:]

    return [
        ajustar(
            janelas[
                int(indice)
            ]
        )
        for indice in sorted(
            indices
        )
    ]


def normalizar_imagem(
    matriz: np.ndarray,
) -> np.ndarray:
    matriz = np.nan_to_num(
        matriz.astype(
            np.float32
        )
    )

    inferior, superior = (
        np.percentile(
            matriz,
            [1, 99],
        )
    )

    normalizada = np.clip(
        (
            matriz - inferior
        )
        / (
            superior
            - inferior
            + 1e-6
        ),
        0,
        1,
    )

    return (
        normalizada * 255.0
    ).astype(
        np.uint8
    )


def criar_feature(
    audio: np.ndarray,
) -> np.ndarray:
    audio = ajustar(
        audio
    )

    mel = (
        librosa.feature
        .melspectrogram(
            y=audio,
            sr=CFG.sr,
            n_fft=CFG.n_fft,
            hop_length=(
                CFG.hop_length
            ),
            n_mels=CFG.n_mels,
            power=2.0,
        )
    )

    log_mel = (
        librosa.power_to_db(
            mel,
            ref=np.max,
            top_db=80.0,
        )
    )

    mfcc = (
        librosa.feature.mfcc(
            S=log_mel,
            n_mfcc=CFG.n_mfcc,
        )
    )

    delta = (
        librosa.feature.delta(
            log_mel,
            width=9,
        )
    )

    canais = []

    for matriz in (
        log_mel,
        mfcc,
        delta,
    ):
        canal = (
            normalizar_imagem(
                matriz
            )
        )

        canal = cv2.resize(
            canal,
            (
                CFG.imagem,
                CFG.imagem,
            ),
            interpolation=(
                cv2.INTER_AREA
            ),
        )

        canais.append(
            canal
        )

    return np.stack(
        canais,
        axis=-1,
    )


if FEATURES.exists():
    shutil.rmtree(
        FEATURES
    )

for split in (
    "treino",
    "validacao",
    "teste",
):
    for classe in CLASSES:
        (
            FEATURES
            / split
            / classe
        ).mkdir(
            parents=True,
            exist_ok=True,
        )

registros_features = []

for linha in tqdm(
    meta_split.itertuples(),
    total=len(meta_split),
    desc="Extraindo features",
):
    try:
        audio = carregar_audio(
            linha.arquivo
        )

        janelas = selecionar_janelas(
            audio,
            linha.classe,
        )

    except Exception as erro:
        print(
            "Falha em:",
            linha.arquivo,
            erro,
        )
        continue

    for (
        indice_janela,
        janela,
    ) in enumerate(janelas):
        imagem = criar_feature(
            janela
        )

        assinatura = hashlib.sha1(
            (
                f"{linha.arquivo}|"
                f"{indice_janela}"
            ).encode("utf-8")
        ).hexdigest()[:18]

        destino = (
            FEATURES
            / linha.split
            / linha.classe
            / f"{assinatura}.png"
        )

        cv2.imwrite(
            str(destino),
            cv2.cvtColor(
                imagem,
                cv2.COLOR_RGB2BGR,
            ),
        )

        registros_features.append(
            {
                "feature": str(
                    destino
                ),
                "classe": (
                    linha.classe
                ),
                "classe_id": (
                    ID[linha.classe]
                ),
                "grupo": linha.grupo,
                "split": linha.split,
                "origem_audio": (
                    linha.arquivo
                ),
                "janela": (
                    indice_janela
                ),
                "fonte": linha.fonte,
            }
        )

features_meta = pd.DataFrame(
    registros_features
)

if features_meta.empty:
    raise RuntimeError(
        "Nenhuma feature foi criada."
    )

features_meta.to_csv(
    RESULTADOS
    / "features_metadata.csv",
    index=False,
)

resumo_features = (
    features_meta.groupby(
        [
            "classe",
            "split",
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

display(resumo_features)

## 8. Criar `tf.data` balanceado

Cada classe possui um dataset próprio, repetido e amostrado com a mesma probabilidade.

Isso corrige o desbalanceamento sem criar arrays gigantes nem duplicar milhares de arquivos.

In [ ]:
def carregar_png(
    caminho,
    rotulo,
):
    imagem = tf.io.read_file(
        caminho
    )

    imagem = tf.io.decode_png(
        imagem,
        channels=3,
    )

    imagem = tf.cast(
        imagem,
        tf.float32,
    )

    return imagem, rotulo


def augmentar_feature(
    imagem,
    rotulo,
):
    imagem = tf.image.random_contrast(
        imagem,
        lower=0.82,
        upper=1.18,
    )

    imagem = tf.image.random_brightness(
        imagem,
        max_delta=10.0,
    )

    deslocamento = tf.random.uniform(
        shape=(),
        minval=-6,
        maxval=7,
        dtype=tf.int32,
    )

    imagem = tf.roll(
        imagem,
        shift=deslocamento,
        axis=1,
    )

    altura = tf.shape(
        imagem
    )[0]
    largura = tf.shape(
        imagem
    )[1]

    tamanho_freq = tf.random.uniform(
        shape=(),
        minval=0,
        maxval=14,
        dtype=tf.int32,
    )
    inicio_freq = tf.random.uniform(
        shape=(),
        minval=0,
        maxval=tf.maximum(
            1,
            altura
            - tamanho_freq,
        ),
        dtype=tf.int32,
    )

    mascara_freq = (
        (
            tf.range(altura)
            >= inicio_freq
        )
        & (
            tf.range(altura)
            < inicio_freq
            + tamanho_freq
        )
    )

    imagem = tf.where(
        mascara_freq[
            :,
            tf.newaxis,
            tf.newaxis,
        ],
        tf.zeros_like(
            imagem
        ),
        imagem,
    )

    tamanho_tempo = tf.random.uniform(
        shape=(),
        minval=0,
        maxval=16,
        dtype=tf.int32,
    )
    inicio_tempo = tf.random.uniform(
        shape=(),
        minval=0,
        maxval=tf.maximum(
            1,
            largura
            - tamanho_tempo,
        ),
        dtype=tf.int32,
    )

    mascara_tempo = (
        (
            tf.range(largura)
            >= inicio_tempo
        )
        & (
            tf.range(largura)
            < inicio_tempo
            + tamanho_tempo
        )
    )

    imagem = tf.where(
        mascara_tempo[
            tf.newaxis,
            :,
            tf.newaxis,
        ],
        tf.zeros_like(
            imagem
        ),
        imagem,
    )

    imagem = tf.clip_by_value(
        imagem,
        0.0,
        255.0,
    )

    return imagem, rotulo


def dataset_finito(
    tabela: pd.DataFrame,
    lote: int,
):
    caminhos = tabela[
        "feature"
    ].astype(str).to_numpy()

    rotulos = tabela[
        "classe_id"
    ].astype(
        np.int32
    ).to_numpy()

    dataset = (
        tf.data.Dataset
        .from_tensor_slices(
            (
                caminhos,
                rotulos,
            )
        )
    )

    dataset = dataset.map(
        carregar_png,
        num_parallel_calls=2,
    )

    dataset = dataset.batch(
        lote
    )

    dataset = dataset.prefetch(
        1
    )

    return dataset


def dataset_balanceado(
    tabela: pd.DataFrame,
    lote: int,
):
    datasets_classes = []

    for classe in CLASSES:
        parte = tabela[
            tabela["classe"]
            == classe
        ]

        if parte.empty:
            raise RuntimeError(
                f"Treino sem {classe}."
            )

        caminhos = parte[
            "feature"
        ].astype(str).to_numpy()

        rotulos = parte[
            "classe_id"
        ].astype(
            np.int32
        ).to_numpy()

        dataset_classe = (
            tf.data.Dataset
            .from_tensor_slices(
                (
                    caminhos,
                    rotulos,
                )
            )
        )

        dataset_classe = (
            dataset_classe
            .shuffle(
                min(
                    len(parte),
                    3000,
                ),
                seed=(
                    CFG.seed
                    + ID[classe]
                ),
                reshuffle_each_iteration=True,
            )
            .repeat()
            .map(
                carregar_png,
                num_parallel_calls=2,
            )
            .map(
                augmentar_feature,
                num_parallel_calls=2,
            )
        )

        datasets_classes.append(
            dataset_classe
        )

    dataset = (
        tf.data.Dataset
        .sample_from_datasets(
            datasets_classes,
            weights=[
                1.0 / len(CLASSES)
            ]
            * len(CLASSES),
            seed=CFG.seed,
            stop_on_empty_dataset=False,
        )
    )

    dataset = dataset.batch(
        lote
    )

    dataset = dataset.prefetch(
        1
    )

    return dataset


meta_treino = (
    features_meta[
        features_meta["split"]
        == "treino"
    ]
    .reset_index(drop=True)
)

meta_validacao = (
    features_meta[
        features_meta["split"]
        == "validacao"
    ]
    .reset_index(drop=True)
)

meta_teste = (
    features_meta[
        features_meta["split"]
        == "teste"
    ]
    .reset_index(drop=True)
)

ds_treino_cabeca = (
    dataset_balanceado(
        meta_treino,
        CFG.lote_cabeca,
    )
)

ds_treino_ajuste = (
    dataset_balanceado(
        meta_treino,
        CFG.lote_ajuste,
    )
)

ds_validacao = dataset_finito(
    meta_validacao,
    CFG.lote_validacao,
)

ds_teste = dataset_finito(
    meta_teste,
    CFG.lote_validacao,
)

passos_epoca = int(
    np.clip(
        math.ceil(
            len(meta_treino)
            / CFG.lote_cabeca
        ),
        CFG.passos_minimos_epoca,
        CFG.passos_maximos_epoca,
    )
)

passos_ajuste = int(
    np.clip(
        math.ceil(
            len(meta_treino)
            / CFG.lote_ajuste
        ),
        CFG.passos_minimos_epoca,
        CFG.passos_maximos_epoca,
    )
)

print(
    "Features:",
    {
        "treino": len(meta_treino),
        "validacao": (
            len(meta_validacao)
        ),
        "teste": len(meta_teste),
    },
)

print(
    "Passos:",
    {
        "cabeca": passos_epoca,
        "ajuste": passos_ajuste,
    },
)

## 9. Modelo

In [ ]:
@tf.keras.utils.register_keras_serializable(
    package="ViolenceEdgeAI"
)
class SparseMacroF1(
    tf.keras.metrics.Metric
):
    def __init__(
        self,
        num_classes: int,
        name: str = "macro_f1",
        **kwargs,
    ):
        super().__init__(
            name=name,
            **kwargs,
        )

        self.num_classes = int(
            num_classes
        )

        self.matriz = (
            self.add_weight(
                name=(
                    "matriz_confusao"
                ),
                shape=(
                    self.num_classes,
                    self.num_classes,
                ),
                initializer="zeros",
                dtype=tf.float32,
            )
        )

    def update_state(
        self,
        y_true,
        y_pred,
        sample_weight=None,
    ):
        y_true = tf.cast(
            tf.reshape(
                y_true,
                [-1],
            ),
            tf.int32,
        )

        y_pred = tf.argmax(
            y_pred,
            axis=-1,
            output_type=tf.int32,
        )

        matriz_lote = (
            tf.math.confusion_matrix(
                y_true,
                y_pred,
                num_classes=(
                    self.num_classes
                ),
                dtype=tf.float32,
            )
        )

        self.matriz.assign_add(
            matriz_lote
        )

    def result(self):
        vp = (
            tf.linalg.diag_part(
                self.matriz
            )
        )

        fp = (
            tf.reduce_sum(
                self.matriz,
                axis=0,
            )
            - vp
        )

        fn = (
            tf.reduce_sum(
                self.matriz,
                axis=1,
            )
            - vp
        )

        precisao = (
            tf.math.divide_no_nan(
                vp,
                vp + fp,
            )
        )

        revocacao = (
            tf.math.divide_no_nan(
                vp,
                vp + fn,
            )
        )

        f1 = (
            tf.math.divide_no_nan(
                2.0
                * precisao
                * revocacao,
                precisao
                + revocacao,
            )
        )

        presentes = tf.cast(
            tf.reduce_sum(
                self.matriz,
                axis=1,
            ) > 0,
            tf.float32,
        )

        return (
            tf.math.divide_no_nan(
                tf.reduce_sum(
                    f1 * presentes
                ),
                tf.reduce_sum(
                    presentes
                ),
            )
        )

    def reset_state(self):
        self.matriz.assign(
            tf.zeros_like(
                self.matriz
            )
        )

    def get_config(self):
        config = (
            super().get_config()
        )

        config.update(
            {
                "num_classes": (
                    self.num_classes
                )
            }
        )

        return config


def criar_modelo():
    tf.keras.backend.clear_session()

    base = (
        tf.keras.applications
        .MobileNetV3Small(
            input_shape=(
                CFG.imagem,
                CFG.imagem,
                3,
            ),
            include_top=False,
            weights="imagenet",
            include_preprocessing=True,
        )
    )

    base.trainable = False

    entrada = tf.keras.Input(
        shape=(
            CFG.imagem,
            CFG.imagem,
            3,
        ),
        dtype=tf.float32,
        name="audio_features",
    )

    x = base(
        entrada,
        training=False,
    )

    x = (
        tf.keras.layers
        .GlobalAveragePooling2D()
        (x)
    )

    x = tf.keras.layers.Dropout(
        0.35
    )(x)

    x = tf.keras.layers.Dense(
        192,
        activation="swish",
    )(x)

    x = (
        tf.keras.layers
        .BatchNormalization()
        (x)
    )

    x = tf.keras.layers.Dropout(
        0.30
    )(x)

    saida = (
        tf.keras.layers.Dense(
            len(CLASSES),
            activation="softmax",
            dtype=tf.float32,
            name="evento",
        )
        (x)
    )

    modelo = tf.keras.Model(
        entrada,
        saida,
        name="aed_v4_mobilenetv3",
    )

    return modelo, base


def compilar_modelo(
    modelo,
    taxa_aprendizado: float,
):
    modelo.compile(
        optimizer=(
            tf.keras.optimizers
            .AdamW(
                learning_rate=(
                    taxa_aprendizado
                ),
                weight_decay=1e-5,
            )
        ),
        loss=(
            tf.keras.losses
            .SparseCategoricalCrossentropy()
        ),
        metrics=[
            (
                tf.keras.metrics
                .SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ),
            (
                tf.keras.metrics
                .SparseTopKCategoricalAccuracy(
                    k=2,
                    name="top2",
                )
            ),
            SparseMacroF1(
                len(CLASSES)
            ),
        ],
        jit_compile=False,
        run_eagerly=False,
    )


model, base = criar_modelo()

compilar_modelo(
    model,
    8e-4,
)

model.summary()

## 10. Treinamento estável

In [ ]:
PESO_CABECA = (
    MODELOS
    / "aed_v4_cabeca.weights.h5"
)

PESO_AJUSTE = (
    MODELOS
    / "aed_v4_ajuste.weights.h5"
)

MODELO_FINAL = (
    MODELOS
    / "aed_v4_final.keras"
)

callbacks_cabeca = [
    tf.keras.callbacks.ModelCheckpoint(
        PESO_CABECA,
        monitor="val_macro_f1",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_macro_f1",
        mode="max",
        patience=CFG.paciencia,
        restore_best_weights=False,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(
        RESULTADOS
        / "historico_cabeca.csv"
    ),
]

historico_cabeca = model.fit(
    ds_treino_cabeca,
    validation_data=(
        ds_validacao
    ),
    epochs=(
        CFG.epocas_cabeca
    ),
    steps_per_epoch=(
        passos_epoca
    ),
    callbacks=(
        callbacks_cabeca
    ),
    verbose=1,
)

if not PESO_CABECA.exists():
    raise FileNotFoundError(
        "Checkpoint da cabeça não criado."
    )

model.load_weights(
    PESO_CABECA
)

avaliacao_cabeca = (
    model.evaluate(
        ds_validacao,
        return_dict=True,
        verbose=1,
    )
)

print(
    "Validação da cabeça:",
    avaliacao_cabeca,
)

base.trainable = True

for camada in base.layers:
    camada.trainable = False

for camada in base.layers[-28:]:
    if not isinstance(
        camada,
        tf.keras.layers
        .BatchNormalization,
    ):
        camada.trainable = True

compilar_modelo(
    model,
    5e-5,
)

callbacks_ajuste = [
    tf.keras.callbacks.ModelCheckpoint(
        PESO_AJUSTE,
        monitor="val_macro_f1",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_macro_f1",
        mode="max",
        patience=CFG.paciencia,
        restore_best_weights=False,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(
        RESULTADOS
        / "historico_ajuste_fino.csv"
    ),
]

historico_ajuste = model.fit(
    ds_treino_ajuste,
    validation_data=(
        ds_validacao
    ),
    epochs=(
        CFG.epocas_ajuste
    ),
    steps_per_epoch=(
        passos_ajuste
    ),
    callbacks=(
        callbacks_ajuste
    ),
    verbose=1,
)

if not PESO_AJUSTE.exists():
    raise FileNotFoundError(
        "Checkpoint do ajuste não criado."
    )

model.load_weights(
    PESO_AJUSTE
)

avaliacao_ajuste = (
    model.evaluate(
        ds_validacao,
        return_dict=True,
        verbose=1,
    )
)

print(
    "Validação do ajuste:",
    avaliacao_ajuste,
)

f1_cabeca = float(
    avaliacao_cabeca[
        "macro_f1"
    ]
)

f1_ajuste = float(
    avaliacao_ajuste[
        "macro_f1"
    ]
)

if f1_cabeca > f1_ajuste:
    model.load_weights(
        PESO_CABECA
    )
    etapa_selecionada = (
        "cabeca"
    )
    melhor_f1_validacao = (
        f1_cabeca
    )
else:
    model.load_weights(
        PESO_AJUSTE
    )
    etapa_selecionada = (
        "ajuste_fino"
    )
    melhor_f1_validacao = (
        f1_ajuste
    )

model.save(
    MODELO_FINAL
)

print(
    "Etapa selecionada:",
    etapa_selecionada,
)
print(
    "Macro-F1 de validação:",
    melhor_f1_validacao,
)
print(
    "Modelo final:",
    MODELO_FINAL,
)

gc.collect()

## 11. Avaliação por janela e gravação

In [ ]:
probabilidades = model.predict(
    ds_teste,
    verbose=1,
)

y_teste = (
    meta_teste[
        "classe_id"
    ]
    .astype(int)
    .to_numpy()
)

previsoes = np.argmax(
    probabilidades,
    axis=1,
)

metricas_janela = {
    "accuracy": float(
        accuracy_score(
            y_teste,
            previsoes,
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(
            y_teste,
            previsoes,
        )
    ),
    "f1_macro": float(
        f1_score(
            y_teste,
            previsoes,
            average="macro",
        )
    ),
    "f1_weighted": float(
        f1_score(
            y_teste,
            previsoes,
            average="weighted",
        )
    ),
}

print(
    json.dumps(
        metricas_janela,
        indent=2,
    )
)

relatorio = (
    classification_report(
        y_teste,
        previsoes,
        target_names=CLASSES,
        output_dict=True,
        digits=4,
    )
)

relatorio_df = (
    pd.DataFrame(
        relatorio
    ).T
)

display(
    relatorio_df
)

relatorio_df.to_csv(
    RESULTADOS
    / "metricas_janela.csv"
)

matriz = confusion_matrix(
    y_teste,
    previsoes,
)

plt.figure(
    figsize=(10, 8)
)
plt.imshow(
    matriz,
    cmap="Blues",
)
plt.title(
    "Matriz de confusão — AED v4"
)
plt.colorbar()
plt.xticks(
    range(len(CLASSES)),
    CLASSES,
    rotation=45,
    ha="right",
)
plt.yticks(
    range(len(CLASSES)),
    CLASSES,
)

for linha in range(
    len(CLASSES)
):
    for coluna in range(
        len(CLASSES)
    ):
        plt.text(
            coluna,
            linha,
            str(
                matriz[
                    linha,
                    coluna,
                ]
            ),
            ha="center",
            va="center",
        )

plt.xlabel("Previsto")
plt.ylabel("Real")
plt.tight_layout()
plt.savefig(
    RESULTADOS
    / "matriz_confusao.png",
    dpi=170,
)
plt.show()

predicoes_df = (
    meta_teste.copy()
)

predicoes_df[
    "real_id"
] = y_teste

for (
    indice,
    classe,
) in enumerate(CLASSES):
    predicoes_df[
        f"prob_{classe}"
    ] = probabilidades[
        :,
        indice,
    ]

linhas_gravacao = []

for (
    grupo,
    parte,
) in predicoes_df.groupby(
    "grupo"
):
    vetor = np.asarray(
        [
            parte[
                f"prob_{classe}"
            ].mean()
            for classe in CLASSES
        ]
    )

    real = int(
        parte[
            "real_id"
        ].mode().iloc[0]
    )

    previsto = int(
        np.argmax(vetor)
    )

    linhas_gravacao.append(
        {
            "grupo": grupo,
            "real": real,
            "previsto": previsto,
            "real_nome": (
                CLASSES[real]
            ),
            "previsto_nome": (
                CLASSES[previsto]
            ),
            "confianca": float(
                np.max(vetor)
            ),
        }
    )

gravacoes = pd.DataFrame(
    linhas_gravacao
)

metricas_gravacao = {
    "accuracy": float(
        accuracy_score(
            gravacoes["real"],
            gravacoes[
                "previsto"
            ],
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(
            gravacoes["real"],
            gravacoes[
                "previsto"
            ],
        )
    ),
    "f1_macro": float(
        f1_score(
            gravacoes["real"],
            gravacoes[
                "previsto"
            ],
            average="macro",
        )
    ),
    "f1_weighted": float(
        f1_score(
            gravacoes["real"],
            gravacoes[
                "previsto"
            ],
            average="weighted",
        )
    ),
}

print(
    "Por gravação:",
    json.dumps(
        metricas_gravacao,
        indent=2,
    ),
)

gravacoes.to_csv(
    RESULTADOS
    / "predicoes_por_gravacao.csv",
    index=False,
)

pd.DataFrame(
    [
        {
            "nivel": "janela",
            **metricas_janela,
        },
        {
            "nivel": "gravacao",
            **metricas_gravacao,
        },
    ]
).to_csv(
    RESULTADOS
    / "resumo_metricas.csv",
    index=False,
)

## 12. Limiares e falsos alertas

In [ ]:
probabilidades_validacao = (
    model.predict(
        ds_validacao,
        verbose=0,
    )
)

y_validacao = (
    meta_validacao[
        "classe_id"
    ]
    .astype(int)
    .to_numpy()
)

limiares = {}
registros_limiares = []

mascara_normal_validacao = (
    y_validacao
    == ID[
        "ambiente_normal"
    ]
)

for (
    indice,
    classe,
) in enumerate(CLASSES):
    if (
        classe
        == "ambiente_normal"
    ):
        limiares[classe] = 0.50
        registros_limiares.append(
            {
                "classe": classe,
                "limiar": 0.50,
                "f1_validacao": float(
                    f1_score(
                        (
                            y_validacao
                            == indice
                        ).astype(int),
                        (
                            probabilidades_validacao[
                                :,
                                indice,
                            ]
                            >= 0.50
                        ).astype(int),
                        zero_division=0,
                    )
                ),
                "taxa_fp_normal": 0.0,
            }
        )
        continue

    real = (
        y_validacao
        == indice
    ).astype(int)

    candidatos = []

    for limiar in np.arange(
        0.35,
        0.96,
        0.01,
    ):
        previsto = (
            probabilidades_validacao[
                :,
                indice,
            ]
            >= limiar
        ).astype(int)

        valor_f1 = f1_score(
            real,
            previsto,
            zero_division=0,
        )

        falsos_normais = (
            previsto[
                mascara_normal_validacao
            ].mean()
            if (
                mascara_normal_validacao
                .any()
            )
            else 0.0
        )

        precisao = precision_score(
            real,
            previsto,
            zero_division=0,
        )

        revocacao = recall_score(
            real,
            previsto,
            zero_division=0,
        )

        objetivo = (
            valor_f1
            - 0.50
            * falsos_normais
        )

        candidatos.append(
            {
                "limiar": float(
                    limiar
                ),
                "f1": float(
                    valor_f1
                ),
                "fp_normal": float(
                    falsos_normais
                ),
                "precisao": float(
                    precisao
                ),
                "revocacao": float(
                    revocacao
                ),
                "objetivo": float(
                    objetivo
                ),
            }
        )

    aceitaveis = [
        item
        for item in candidatos
        if (
            item["precisao"]
            >= 0.70
            and item[
                "fp_normal"
            ] <= 0.10
        )
    ]

    if aceitaveis:
        melhor = max(
            aceitaveis,
            key=lambda item: (
                item["revocacao"],
                item["f1"],
            ),
        )
    else:
        melhor = max(
            candidatos,
            key=lambda item: (
                item["objetivo"]
            ),
        )

    limiares[classe] = round(
        max(
            0.55,
            melhor["limiar"],
        ),
        2,
    )

    registros_limiares.append(
        {
            "classe": classe,
            "limiar": (
                limiares[classe]
            ),
            "f1_validacao": (
                melhor["f1"]
            ),
            "taxa_fp_normal": (
                melhor[
                    "fp_normal"
                ]
            ),
            "precisao": (
                melhor["precisao"]
            ),
            "revocacao": (
                melhor["revocacao"]
            ),
        }
    )

tabela_limiares = (
    pd.DataFrame(
        registros_limiares
    )
)

display(
    tabela_limiares
)

tabela_limiares.to_csv(
    RESULTADOS
    / "limiares.csv",
    index=False,
)

(
    MODELOS
    / "limiares.json"
).write_text(
    json.dumps(
        limiares,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

mascara_normal_teste = (
    y_teste
    == ID[
        "ambiente_normal"
    ]
)

falsos_alertas = 0

for vetor in probabilidades[
    mascara_normal_teste
]:
    indice = int(
        np.argmax(vetor)
    )
    classe = CLASSES[
        indice
    ]

    ativo = (
        classe
        != "ambiente_normal"
        and vetor[indice]
        >= limiares[classe]
    )

    falsos_alertas += int(
        ativo
    )

janelas_normais = int(
    mascara_normal_teste.sum()
)

horas = (
    janelas_normais
    * CFG.duracao
    / 3600.0
)

falsos_por_hora = (
    falsos_alertas / horas
    if horas > 0
    else 0.0
)

auditoria_falsos = {
    "janelas_normais": (
        janelas_normais
    ),
    "horas_estimadas": (
        horas
    ),
    "falsos_alertas": (
        falsos_alertas
    ),
    "falsos_alertas_por_hora": (
        falsos_por_hora
    ),
    "atingiu_alvo": (
        falsos_por_hora
        <= CFG.max_falsos_alertas_hora
    ),
}

print(
    json.dumps(
        auditoria_falsos,
        indent=2,
    )
)

(
    RESULTADOS
    / "auditoria_falsos_alertas.json"
).write_text(
    json.dumps(
        auditoria_falsos,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

## 13. Exportar e verificar TFLite

In [ ]:
TFLITE_FLOAT32 = (
    MODELOS
    / "aed_v4_float32.tflite"
)

TFLITE_FLOAT16 = (
    MODELOS
    / "aed_v4_float16.tflite"
)

TFLITE_INT8 = (
    MODELOS
    / "aed_v4_int8.tflite"
)

conversor = (
    tf.lite.TFLiteConverter
    .from_keras_model(model)
)

TFLITE_FLOAT32.write_bytes(
    conversor.convert()
)

conversor = (
    tf.lite.TFLiteConverter
    .from_keras_model(model)
)

conversor.optimizations = [
    tf.lite.Optimize.DEFAULT
]

conversor.target_spec.supported_types = [
    tf.float16
]

TFLITE_FLOAT16.write_bytes(
    conversor.convert()
)

ds_representativo = (
    dataset_finito(
        meta_treino.sample(
            min(
                400,
                len(meta_treino),
            ),
            random_state=CFG.seed,
        ),
        lote=1,
    )
)

def dados_representativos():
    contador = 0

    for imagens, _ in (
        ds_representativo
    ):
        yield [
            tf.cast(
                imagens,
                tf.float32,
            )
        ]

        contador += 1

        if contador >= 300:
            return

try:
    conversor = (
        tf.lite.TFLiteConverter
        .from_keras_model(model)
    )

    conversor.optimizations = [
        tf.lite.Optimize.DEFAULT
    ]

    conversor.representative_dataset = (
        dados_representativos
    )

    conversor.target_spec.supported_ops = [
        (
            tf.lite.OpsSet
            .TFLITE_BUILTINS_INT8
        )
    ]

    conversor.inference_input_type = (
        tf.int8
    )
    conversor.inference_output_type = (
        tf.int8
    )

    TFLITE_INT8.write_bytes(
        conversor.convert()
    )

    print("INT8 exportado.")

except Exception as erro:
    print(
        "INT8 não exportado:",
        erro,
    )

for caminho in (
    MODELO_FINAL,
    TFLITE_FLOAT32,
    TFLITE_FLOAT16,
    TFLITE_INT8,
):
    if caminho.exists():
        print(
            caminho.name,
            (
                caminho.stat().st_size
                / 1e6
            ),
            "MB",
        )

In [ ]:
def prever_tflite(
    caminho: Path,
    imagens: np.ndarray,
) -> np.ndarray:
    interpretador = (
        tf.lite.Interpreter(
            model_path=str(
                caminho
            )
        )
    )

    interpretador.allocate_tensors()

    entrada = (
        interpretador
        .get_input_details()[0]
    )
    saida = (
        interpretador
        .get_output_details()[0]
    )

    resultados = []

    for imagem in imagens:
        valor = imagem[
            np.newaxis,
            ...,
        ].astype(
            np.float32
        )

        if entrada["dtype"] in (
            np.int8,
            np.uint8,
        ):
            escala, zero = (
                entrada[
                    "quantization"
                ]
            )

            valor = np.round(
                valor / escala
                + zero
            )

            limites = np.iinfo(
                entrada["dtype"]
            )

            valor = np.clip(
                valor,
                limites.min,
                limites.max,
            ).astype(
                entrada["dtype"]
            )
        else:
            valor = valor.astype(
                entrada["dtype"]
            )

        interpretador.set_tensor(
            entrada["index"],
            valor,
        )

        interpretador.invoke()

        predicao = (
            interpretador.get_tensor(
                saida["index"]
            )
        )

        if saida["dtype"] in (
            np.int8,
            np.uint8,
        ):
            escala, zero = (
                saida[
                    "quantization"
                ]
            )

            predicao = (
                predicao.astype(
                    np.float32
                )
                - zero
            ) * escala

        resultados.append(
            predicao[0]
        )

    return np.asarray(
        resultados
    )


amostra_teste = (
    meta_teste.sample(
        min(
            250,
            len(meta_teste),
        ),
        random_state=CFG.seed,
    )
)

imagens_amostra = []

for caminho in amostra_teste[
    "feature"
]:
    imagem = cv2.imread(
        str(caminho),
        cv2.IMREAD_COLOR,
    )

    imagem = cv2.cvtColor(
        imagem,
        cv2.COLOR_BGR2RGB,
    )

    imagens_amostra.append(
        imagem.astype(
            np.float32
        )
    )

imagens_amostra = np.asarray(
    imagens_amostra
)

pred_keras = model.predict(
    imagens_amostra,
    batch_size=32,
    verbose=0,
)

registros_paridade = []

for caminho in (
    TFLITE_FLOAT32,
    TFLITE_FLOAT16,
    TFLITE_INT8,
):
    if not caminho.exists():
        continue

    pred_tflite = prever_tflite(
        caminho,
        imagens_amostra,
    )

    concordancia = float(
        np.mean(
            np.argmax(
                pred_tflite,
                axis=1,
            )
            == np.argmax(
                pred_keras,
                axis=1,
            )
        )
    )

    diferenca = float(
        np.max(
            np.abs(
                pred_tflite
                - pred_keras
            )
        )
    )

    registros_paridade.append(
        {
            "modelo": (
                caminho.name
            ),
            "concordancia_argmax": (
                concordancia
            ),
            "diferenca_maxima": (
                diferenca
            ),
        }
    )

paridade = pd.DataFrame(
    registros_paridade
)

display(paridade)

paridade.to_csv(
    RESULTADOS
    / "paridade_tflite.csv",
    index=False,
)

modelos_float = paridade[
    paridade["modelo"].isin(
        [
            TFLITE_FLOAT32.name,
            TFLITE_FLOAT16.name,
        ]
    )
]

if (
    not modelos_float.empty
    and modelos_float[
        "concordancia_argmax"
    ].min()
    < CFG.meta_paridade_tflite
):
    raise RuntimeError(
        "Paridade Float32/Float16 "
        "abaixo da meta."
    )

## 14. Teste externo e VAD paralelo

In [ ]:
enviados = files.upload()

arquivos_audio = [
    nome
    for nome in enviados
    if Path(
        nome
    ).suffix.lower()
    in EXTENSOES_AUDIO
]

if len(arquivos_audio) != 1:
    raise RuntimeError(
        "Envie exatamente um áudio."
    )

AUDIO_EXTERNO = Path(
    arquivos_audio[0]
).resolve()

display(
    Audio(
        str(AUDIO_EXTERNO)
    )
)


def janelas_deslizantes(
    audio: np.ndarray,
):
    passo = int(
        CFG.sr
        * CFG.passo_inferencia
    )

    if len(audio) <= AMOSTRAS:
        return [
            (
                0.0,
                ajustar(audio),
            )
        ]

    return [
        (
            inicio / CFG.sr,
            audio[
                inicio:
                inicio + AMOSTRAS
            ],
        )
        for inicio in range(
            0,
            len(audio)
            - AMOSTRAS
            + 1,
            passo,
        )
    ]


def inferir_audio(
    caminho: Path,
):
    audio = carregar_audio(
        caminho
    )

    janelas = (
        janelas_deslizantes(
            audio
        )
    )

    imagens = np.asarray(
        [
            criar_feature(
                janela
            )
            for _, janela
            in janelas
        ],
        dtype=np.float32,
    )

    probs = model.predict(
        imagens,
        batch_size=32,
        verbose=0,
    )

    linhas = []

    for (
        inicio,
        _,
    ), vetor in zip(
        janelas,
        probs,
        strict=True,
    ):
        indice = int(
            np.argmax(vetor)
        )
        classe = CLASSES[
            indice
        ]
        confianca = float(
            vetor[indice]
        )

        ativo = (
            classe
            != "ambiente_normal"
            and confianca
            >= limiares[classe]
        )

        linhas.append(
            {
                "inicio": inicio,
                "fim": (
                    inicio
                    + CFG.duracao
                ),
                "classe": classe,
                "confianca": (
                    confianca
                ),
                "limiar": (
                    limiares[
                        classe
                    ]
                ),
                "ativo": ativo,
            }
        )

    tabela = pd.DataFrame(
        linhas
    )

    consecutivas = 0
    classe_anterior = None
    alertas = []

    for linha in (
        tabela.itertuples()
    ):
        if (
            linha.ativo
            and linha.classe
            == classe_anterior
        ):
            consecutivas += 1

        elif linha.ativo:
            consecutivas = 1

        else:
            consecutivas = 0

        classe_anterior = (
            linha.classe
            if linha.ativo
            else None
        )

        alertas.append(
            consecutivas
            >= CFG.minimo_consecutivas_alerta
        )

    tabela[
        "alerta_temporal"
    ] = alertas

    return tabela


timeline = inferir_audio(
    AUDIO_EXTERNO
)

display(
    timeline.sort_values(
        "confianca",
        ascending=False,
    ).head(30)
)

timeline.to_csv(
    RESULTADOS
    / "timeline_audio_externo.csv",
    index=False,
)


def executar_vad(
    caminho: Path,
):
    audio = carregar_audio(
        caminho
    )

    pcm = (
        np.clip(
            audio,
            -1.0,
            1.0,
        )
        * 32767
    ).astype(
        np.int16
    )

    vad = webrtcvad.Vad(
        2
    )

    frame_ms = 30
    tamanho = int(
        CFG.sr
        * frame_ms
        / 1000
    )

    decisoes = []

    for inicio in range(
        0,
        len(pcm)
        - tamanho
        + 1,
        tamanho,
    ):
        frame = pcm[
            inicio:
            inicio + tamanho
        ]

        decisoes.append(
            vad.is_speech(
                frame.tobytes(),
                CFG.sr,
            )
        )

    return {
        "frames": len(
            decisoes
        ),
        "frames_fala": int(
            sum(decisoes)
        ),
        "proporcao_fala": (
            float(
                np.mean(
                    decisoes
                )
            )
            if decisoes
            else 0.0
        ),
        "bloqueia_aed": False,
    }


print(
    "VAD:",
    executar_vad(
        AUDIO_EXTERNO
    ),
)

## 15. Pacote final

In [ ]:
atingiu_meta = (
    metricas_janela[
        "accuracy"
    ]
    >= CFG.meta_acuracia
    and metricas_janela[
        "f1_macro"
    ]
    >= CFG.meta_f1_macro
)

configuracao_final = {
    "configuracao": asdict(
        CFG
    ),
    "classes": CLASSES,
    "limiares": limiares,
    "etapa_selecionada": (
        etapa_selecionada
    ),
    "melhor_f1_validacao": (
        melhor_f1_validacao
    ),
    "metricas_janela": (
        metricas_janela
    ),
    "metricas_gravacao": (
        metricas_gravacao
    ),
    "auditoria_falsos": (
        auditoria_falsos
    ),
    "atingiu_meta": (
        atingiu_meta
    ),
    "vad": {
        "bloqueia_aed": False,
        "uso": (
            "Acionar Whisper e NLP"
        ),
    },
}

(
    MODELOS
    / "configuracao_aed_v4.json"
).write_text(
    json.dumps(
        configuracao_final,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

if ENTREGA.exists():
    shutil.rmtree(
        ENTREGA
    )

ENTREGA.mkdir(
    parents=True,
    exist_ok=True,
)

for arquivo in MODELOS.glob(
    "*"
):
    if arquivo.is_file():
        shutil.copy2(
            arquivo,
            ENTREGA
            / arquivo.name,
        )

for arquivo in RESULTADOS.glob(
    "*"
):
    if arquivo.is_file():
        shutil.copy2(
            arquivo,
            ENTREGA
            / arquivo.name,
        )

ZIP_FINAL = shutil.make_archive(
    "/content/aed_v4_entrega",
    "zip",
    root_dir=str(
        ENTREGA
    ),
)

print(
    "Atingiu a meta:",
    atingiu_meta,
)
print(
    "ZIP:",
    ZIP_FINAL,
)

files.download(
    ZIP_FINAL
)

## Critério de aceitação

O modelo só deve seguir para a Raspberry Pi quando:

- os rótulos do Scream Dataset estiverem distribuídos entre `ambiente_normal` e `grito_panico`;
- não houver vazamento por grupo;
- nenhuma época mostrar `accuracy=0` por esgotamento do dataset;
- acurácia, F1 macro e métricas por classe forem aceitáveis;
- falsos alertas em áudios realmente normais forem baixos;
- TFLite Float32 e Float16 apresentarem pelo menos 99% de concordância com Keras;
- áudios externos gravados pelo INMP441 forem testados.